In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

from utils.utils import evaluate_models, coordinate_imputer, convert_to_day_of_year
from utils.constants import latitude, longitude, is_coastal

import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error



d:\Programowanie\ryanair_recruitment_task\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#full_pipeline_and_evaluation(df, 'ActualTOW', latitude, longitude, is_coastal)

In [3]:
path_to_training_file = os.path.join('datasets', 'training.csv')

df = pd.read_csv(path_to_training_file, sep='\t')

path_to_validation_file = os.path.join('datasets', 'validation.csv')

df_validation = pd.read_csv(path_to_validation_file, sep='\t')

In [4]:
numeric_values = ['FlightNumber', 'ActualFlightTime', 'ActualTotalFuel', 'FLownPassengers', 
                  'BagsCount', 'FlightBagsWeight']


numeric_values_post_mapping = ['FlightNumber', 'ActualFlightTime', 'ActualTotalFuel', 'FLownPassengers', 
                  'BagsCount', 'FlightBagsWeight', 'DepartureAirportLatitude', 'DepartureAirportLongitude',
                  'ArrivalAirportLatitude', 'ArrivalAirportLongitude']

In [5]:
columns_to_convert = ['ActualTOW','FLownPassengers','BagsCount','FlightBagsWeight']
for i in columns_to_convert:
    df[f'{i}']=pd.to_numeric(df[f'{i}'], errors='coerce')

In [6]:
def simple_pipeline(df):

    df = coordinate_imputer(df, "DepartureAirport", latitude, longitude)
    df = coordinate_imputer(df, "ArrivalAirport", latitude, longitude)
    df['day_of_year'] = df.apply(convert_to_day_of_year, axis=1, args=("DepartureYear", "DepartureMonth", "DepartureDay"))
    df['IsCoastalArrival'] = df['ArrivalAirport'].map(is_coastal)
    df['IsCoastalDeparture'] = df['DepartureAirport'].map(is_coastal)

    df = df.drop(columns=["ArrivalAirport", "DepartureAirport",
            "DepartureMonth", "DepartureDay",
            "DepartureDate", "Route", "DepartureYear"])
    return df

In [7]:
df = df.query("ActualFlightTime < 500")
df = df.query("ActualTotalFuel < 16000")
df = df.query("BagsCount < 200")
df = df.query("FlightBagsWeight < 2500")

df = df.dropna()

In [8]:
df = simple_pipeline(df)

In [9]:
numeric_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])


In [10]:
X_temp, X_test, y_temp, y_test = train_test_split(df.drop(['ActualTOW'], axis=1), df['ActualTOW'], test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

In [11]:
X_prepared = numeric_pipeline.fit_transform(X_train)
X_test_transformed = numeric_pipeline.transform(X_test)
X_valid_transformed = numeric_pipeline.transform(X_valid)


In [12]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-8, 10.0),
        'random_state': 42,
        'tree_method': 'hist',  # szybkie trenowanie
        'verbosity': 0
    }
    
    model = XGBRegressor(**param)
    
    # Używamy cross_val_score (5-krotna walidacja)
    scores = cross_val_score(
        model, X_prepared, y_train,
        scoring='neg_root_mean_squared_error',
        cv=5,
        n_jobs=-1
    )
    
    # Zwracamy średni RMSE (ujemny, bo sklearn liczy jako negatyw)
    return -np.mean(scores)

# Tworzymy obiekt study, który optymalizuje minimalizację (tu RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5, timeout=600)  # np 50 prób lub 10 minut

print("Najlepsze parametry:", study.best_params)
print("Najlepszy RMSE:", study.best_value)

[I 2025-05-15 02:37:22,565] A new study created in memory with name: no-name-c67d5871-fb1f-45f1-bda7-207debbac6bd
C:\Users\ZBlas\AppData\Local\Temp\ipykernel_23604\1178550450.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
C:\Users\ZBlas\AppData\Local\Temp\ipykernel_23604\1178550450.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-8, 10.0),
C:\Users\ZBlas\AppData\Local\Temp\ipykernel_23604\1178550450.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/o

Najlepsze parametry: {'n_estimators': 652, 'max_depth': 8, 'learning_rate': 0.02948395566904586, 'subsample': 0.5568237093636655, 'colsample_bytree': 0.9804785153735942, 'reg_alpha': 1.9918029972454683e-08, 'reg_lambda': 4.398509733420372e-06}
Najlepszy RMSE: 741.8646774655444


In [13]:
best_params = study.best_params
best_params_yet = {'n_estimators': 877, 'max_depth': 6, 'learning_rate': 0.044946052309561155, 'subsample': 0.5842292180281461, 'colsample_bytree': 0.7389452972457078, 'reg_alpha': 0.03327619465140871, 'reg_lambda': 0.13475773657466886}
model = XGBRegressor(**best_params_yet)
model.fit(X_prepared, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7389452972457078, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.044946052309561155,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=877, n_jobs=None,
             num_parallel_tree=None, ...)

In [14]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test_transformed)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation R2: {r2:.4f}")

Validation RMSE: 523530.1239
Validation R2: 0.9504


In [15]:
model.fit(X_test_transformed, y_test)
y_val_pred = model.predict(X_valid_transformed)
rmse = mean_squared_error(y_valid, y_val_pred)
r2 = r2_score(y_valid, y_val_pred)

print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation R2: {r2:.4f}")

Validation RMSE: 687241.8866
Validation R2: 0.9344


In [16]:
for i in numeric_values:
    df_validation[i] = pd.to_numeric(df_validation[i], errors='coerce')
df_validation_post_pipeline = simple_pipeline(df_validation)
df_validation_transformed = numeric_pipeline.transform(df_validation_post_pipeline)
pd.DataFrame(df_validation_transformed)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,0.364387,-0.536917,-0.566807,0.881673,-0.067588,-0.039028,-0.390720,0.040218,0.321872,-0.977930,5.553882,0.973276,0.980700
1,0.364714,-0.495574,-0.537554,1.032058,0.130649,0.326570,0.337223,-0.965119,-0.405025,0.020255,5.553882,0.973276,0.980700
2,0.540912,-0.206173,-0.196271,0.029494,0.357205,0.404121,-0.390720,0.040218,0.820698,-0.496068,5.553882,-1.027458,0.980700
3,0.540585,-0.206173,-0.050007,0.881673,-0.350784,-0.315997,0.836766,-0.479804,-0.405025,0.020255,5.553882,0.973276,-1.019679
4,0.213407,0.041885,0.052378,-0.020634,-0.492381,-0.482178,0.836766,-0.479804,-0.737648,-0.399521,5.553882,0.973276,-1.019679
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1873,-0.926313,-1.033033,-0.983660,0.280135,-1.164971,-1.146902,-0.955738,-0.218688,-0.832534,-0.889729,5.553882,-1.027458,0.980700
1874,0.501939,-0.226845,-0.327909,0.781417,-0.846376,-0.842237,-0.231414,0.672964,-0.978288,-0.570269,5.553882,0.973276,-1.019679
1875,0.501612,-0.123487,-0.274279,0.731289,-0.492381,-0.482178,-0.964807,-0.554537,-0.245948,0.648500,5.553882,-1.027458,0.980700
1876,-0.188115,-1.053704,-1.034365,0.079622,-1.235770,-1.229993,-0.725056,1.243618,-0.245948,0.648500,5.553882,-1.027458,0.980700


In [17]:
X = df.drop(columns=['ActualTOW'])
y = df['ActualTOW']

df_transformed = numeric_pipeline.transform(X)

In [18]:
production_model = XGBRegressor(**best_params_yet)

In [19]:
production_model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7389452972457078, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.044946052309561155,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=877, n_jobs=None,
             num_parallel_tree=None, ...)

In [20]:
prediction = production_model.predict(df_validation_transformed)

In [ ]:
df_prediction = pd.DataFrame(prediction)


In [25]:
df_prediction.to_csv('prediction.csv', index=False)